# Fire: v2.0.0 minus v1.1.0

Subtract the two rasters and look at where they differ. Run in-region (`pixi run start-coiled-notebook`).

In [ ]:
import icechunk
import xarray as xr

from ocr.risks.fire.grid import ChunkingConfig

OLD = 'output/fire-risk/tensor/production/v1.1.0/ocr.icechunk'
DEV = 'output/fire/v2.0.0/store.icechunk'
REGION = 'y5_x24'  # any region id; see ChunkingConfig().valid_region_ids

In [ ]:
def open_store(prefix: str, group: str | None = None) -> xr.Dataset:
    storage = icechunk.s3_storage(
        bucket='carbonplan-ocr', prefix=prefix, region='us-west-2', from_env=True
    )
    store = icechunk.Repository.open(storage).readonly_session('main').store
    return xr.open_zarr(store, group=group, consolidated=False)


old = open_store(OLD)
dev = open_store(DEV, group='0')[list(old.data_vars)]
diff = dev - old  # lazy
diff

In [ ]:
y, x = ChunkingConfig().region_id_to_latlon_slices(region_id=REGION)
d = diff.sel(latitude=y, longitude=x).compute()

for v in d.data_vars:
    print(f'{v}: nonzero={int((d[v] != 0).sum())}  max|diff|={float(abs(d[v]).max())}')

In [ ]:
var = list(d.data_vars)[0]
d[var].plot(robust=True, cmap='RdBu_r', center=0, figsize=(10, 6))